# 跨城市 × 7情景 分组柱状图（扩容 + 定容 各一张）

数据组合方式（与原 5.py 一致）：
- **2020 Baseline** 取自「现状」策略
- **2040 / 2060 的 RCP 各情景** 分别取自「扩容」和「定容」策略，各画一张图

X 轴使用拼音首字母标签（Bei3jing1s 等），无标题，纯 SCI 风格。

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from config.paths import PER_CAPITA_OUTPUT_DIR, FIGURES_DIR
from src.plotting.grouped_bar import plot_scenario_grouped

In [2]:
df = pd.read_csv(os.path.join(PER_CAPITA_OUTPUT_DIR, 'per_capita_hours_summary.csv'))
df.head()

,策略,城市,城市拼音,情景,Total_Person_Hours,Total_Population,Hours_Per_Resident,城市标签
0,现状,北京市,bei3jing1shi4,2020 Baseline,6.084353e+06,5.900282e+06,1.031197,Bei3jing1s
1,扩容,北京市,bei3jing1shi4,2040 RCP 2.6,1.150382e+07,5.900282e+06,1.949708,Bei3jing1s
2,扩容,北京市,bei3jing1shi4,2040 RCP 4.5,1.608852e+07,5.900282e+06,2.726738,Bei3jing1s
3,扩容,北京市,bei3jing1shi4,2040 RCP 8.5,2.601935e+07,5.900282e+06,4.409849,Bei3jing1s
4,扩容,北京市,bei3jing1shi4,2060 RCP 2.6,3.629018e+07,5.900282e+06,6.150585,Bei3jing1s


In [ ]:
def build_plot_df(strategy_name):
    """组合：现状的 2020 Baseline + 指定策略的所有 RCP 情景，加 Grand Total"""
    df_baseline = df[(df['策略'] == '现状') & (df['情景'] == '2020 Baseline')]
    df_future = df[(df['策略'] == strategy_name) & (df['情景'] != '2020 Baseline')]
    df_cities = pd.concat([df_baseline, df_future], ignore_index=True)

    # 与原版 5.py 一致：Grand Total 用人口加权平均（总人时 / 总人口）
    grand = df_cities.groupby('情景').agg(
        Total_Person_Hours=('Total_Person_Hours', 'sum'),
        Total_Population=('Total_Population', 'sum'),
    ).reset_index()
    grand['Hours_Per_Resident'] = grand['Total_Person_Hours'] / grand['Total_Population']
    grand['城市'] = '总计'
    grand['城市标签'] = 'Grand Total'
    grand['策略'] = '加权'
    grand['城市拼音'] = ''

    return pd.concat([df_cities, grand], ignore_index=True)

df_expansion = build_plot_df('扩容')
df_fixed = build_plot_df('定容')
print(f'扩容版数据: {len(df_expansion)} 行')
print(f'定容版数据: {len(df_fixed)} 行')

In [ ]:
# === 扩容版 ===
save_path_exp = os.path.join(FIGURES_DIR, 'Grouped_BarChart_扩容_SCI_600DPI.png')
fig1, ax1 = plot_scenario_grouped(df_expansion, save_path=save_path_exp, add_grand_total=False)

In [ ]:
# === 定容版 ===
save_path_fix = os.path.join(FIGURES_DIR, 'Grouped_BarChart_定容_SCI_600DPI.png')
fig2, ax2 = plot_scenario_grouped(df_fixed, save_path=save_path_fix, add_grand_total=False)